[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tulin206/erumhub_deep-learning_2026/blob/main/code/06_Building_NN_From_Scratch_Advanced_Exercise.ipynb)

## Introduction

Source : https://www.kaggle.com/code/antmarakis/another-neural-network-from-scratch

In this exercise, you will build a **multi-layer neural network from scratch** using only `numpy`. You will implement forward propagation, back-propagation, weight initialization, training, and evaluation — all by yourself!

Parts marked with `### YOUR CODE HERE ###` need to be filled in. Read the docstrings and surrounding code carefully.

In [ ]:
import kagglehub
uciml_iris_path = kagglehub.dataset_download('uciml/iris')

print('Data source import complete.')

## Data

Before the actual implementation, we need to read and process our data. For the purposes of this tutorial, we will use the famous Iris Species Dataset. You can find more information on the Dataset Page right here on [Kaggle](https://www.kaggle.com/uciml/iris). In short, the dataset contains data for **three** species of a flower and we have **four** pieces of data for each sample (sepal length/width and petal length/width).

The two libraries we will mainly use are `numpy` for the mathematical operations and `pandas` for reading the dataset. Let's import them!

In [ ]:
import numpy as np
import pandas as pd

Next we will read the dataset using `pandas` and shuffle it. Shuffling a dataset is usually a good practice — it helps both when splitting into train/test/validation sets and when avoiding overfitting.

In [ ]:
import os
import kagglehub
import pandas as pd

# Download the latest version of the dataset
uciml_iris_path = kagglehub.dataset_download("uciml/iris")
print("Dataset downloaded to:", uciml_iris_path)

# Load the CSV (file name in this dataset is 'Iris.csv')
iris_csv = os.path.join(uciml_iris_path, "Iris.csv")
iris = pd.read_csv(iris_csv)
iris = iris.sample(frac=1).reset_index(drop=True)  # Shuffle

We need to grab the data (the information on each sample) from the `pandas` array and put it into a `numpy` array.

In [ ]:
X = iris[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
X = np.array(X)
X[:5]

The above will be fed into our neural network for training. Now we need to convert the class labels from categorical (`'Setosa'`, `'Versicolor'`, `'Virginica'`) to numerical (0, 1, 2) and then to **one-hot encoded** format ([1,0,0], [0,1,0], [0,0,1]).

We will use `OneHotEncoder` from `sklearn` for this.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder(sparse_output=False)

Y = iris.Species
Y = one_hot_encoder.fit_transform(np.array(Y).reshape(-1, 1))
Y[:5]

Next we will split our dataset into **train / validation / test** sets using `sklearn`. First we split into train/test, then the training data is further split into train/validation.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.1)

## Implementation

We are going to build a simple neural network that supports multiple layers and validation. The main function is `NeuralNetwork`, which will train the network for the specified number of epochs.

* `X_train`, `Y_train`: Training data and labels.
* `X_val`, `Y_val`: Validation data and labels (optional).
* `epochs`: Number of training epochs. Default: 10.
* `nodes`: A list of integers. Each integer denotes the number of nodes in each layer.
* `lr`: Learning rate. Default: 0.15.

In [ ]:
def NeuralNetwork(X_train, Y_train, X_val=None, Y_val=None, epochs=10, nodes=[], lr=0.15):
    hidden_layers = len(nodes) - 1
    weights = InitializeWeights(nodes)

    for epoch in range(1, epochs+1):
        weights = Train(X_train, Y_train, lr, weights)

        if(epoch % 20 == 0):
            print("Epoch {}".format(epoch))
            print("Training Accuracy:{}".format(Accuracy(X_train, Y_train, weights)))
            if X_val.any():
                print("Validation Accuracy:{}".format(Accuracy(X_val, Y_val, weights)))

    return weights

### ✏️ Exercise 1: Initialize Weights

Complete the `InitializeWeights` function below.

The weights should be initialized with **random values in the range [-1, 1]**. For each layer `i`, create a 2D matrix of shape `(nodes[i], nodes[i-1] + 1)` — the `+1` accounts for the **bias**.

> **Hint:** Use `np.random.uniform(-1, 1)` to generate random values and `np.matrix(...)` to create a matrix.

In [ ]:
def InitializeWeights(nodes):
    """Initialize weights with random values in [-1, 1] (including bias)"""
    layers, weights = len(nodes), []

    for i in range(1, layers):
        ### YOUR CODE HERE ###
        # Create a weight matrix for the connections from layer i-1 to layer i
        # Shape: (nodes[i]) rows x (nodes[i-1] + 1) columns  [+1 for bias]
        # Each weight should be a random float in the range [-1, 1]
        w = None  # Replace this line
        ### END YOUR CODE ###
        weights.append(np.matrix(w))

    return weights

### ✏️ Exercise 2: Forward Propagation

Complete the `ForwardPropagation` function.

**Steps for each layer:**
1. Compute the dot product between `layer_input` and the **transposed** weight matrix for that layer.
2. Pass the result through the `Sigmoid` activation function.
3. Append the activation to the `activations` list.
4. Augment the activation with a **bias of 1** (using `np.append`) for the next layer's input.

> **Hint:** `np.dot(a, b)` computes the dot product. Use `weights[j].T` for the transpose.

In [ ]:
def ForwardPropagation(x, weights, layers):
    activations, layer_input = [x], x
    for j in range(layers):
        ### YOUR CODE HERE ###
        # Step 1: Compute dot product of layer_input and weights[j].T
        # Step 2: Apply Sigmoid activation
        # Step 3: Append activation to activations list
        # Step 4: Update layer_input by appending bias (1) to the activation
        activation = None  # Replace this line
        ### END YOUR CODE ###
        activations.append(activation)
        layer_input = np.append(1, activation)  # Augment with bias

    return activations

### ✏️ Exercise 3: Back Propagation

Complete the `BackPropagation` function.

**Steps (iterating from the last layer back to the first):**
1. Compute the **delta**: element-wise product of `error` and `SigmoidDerivative(currActivation)`.
2. **Update the weights**: add `lr * (delta.T * prevActivation)` to `weights[j-1]`.
3. **Remove the bias column** (column 0) from `weights[j-1]` using `np.delete`.
4. **Propagate the error** backwards: `error = np.dot(delta, w)`.

The initial error is `y - outputFinal` (actual minus predicted).

> **Hint:** Use `np.multiply` for element-wise multiplication and `np.multiply(delta.T, prevActivation)` for the weight update.

In [ ]:
def BackPropagation(y, activations, weights, layers):
    outputFinal = activations[-1]
    error = np.matrix(y - outputFinal)  # Error at output

    for j in range(layers, 0, -1):
        currActivation = activations[j]

        if(j > 1):
            prevActivation = np.append(1, activations[j-1])  # Augment with bias
        else:
            prevActivation = activations[0]  # Input layer (no bias added)

        ### YOUR CODE HERE ###
        # Step 1: Compute delta = error ⊙ SigmoidDerivative(currActivation)
        # Step 2: Update weights[j-1] += lr * (delta.T * prevActivation)
        # Step 3: Remove bias column (index 0) from weights[j-1]
        # Step 4: Propagate error = np.dot(delta, w)
        delta = None  # Replace this line
        ### END YOUR CODE ###

    return weights

The `Train` function passes each training sample through the network (forward then backward pass) and returns the updated weights.

In [ ]:
def Train(X, Y, lr, weights):
    layers = len(weights)
    for i in range(len(X)):
        x, y = X[i], Y[i]
        x = np.matrix(np.append(1, x))  # Augment feature vector with bias

        activations = ForwardPropagation(x, weights, layers)
        weights = BackPropagation(y, activations, weights, layers)

    return weights

### ✏️ Exercise 4: Activation Functions

Implement the **Sigmoid** function and its **derivative**.

- Sigmoid: $\sigma(x) = \dfrac{1}{1 + e^{-x}}$
- Sigmoid Derivative: $\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$  
  *(Note: here `x` is already the sigmoid output, so you can use `x * (1 - x)` directly)*

> **Hint:** Use `np.exp` for the exponential and `np.multiply` for element-wise multiplication.

In [ ]:
def Sigmoid(x):
    ### YOUR CODE HERE ###
    # Implement the sigmoid function: 1 / (1 + e^(-x))
    pass  # Replace this line
    ### END YOUR CODE ###

def SigmoidDerivative(x):
    ### YOUR CODE HERE ###
    # Implement the sigmoid derivative: x * (1 - x)
    # Note: x here is already the sigmoid-activated output
    pass  # Replace this line
    ### END YOUR CODE ###

### ✏️ Exercise 5: Predict

Complete the `Predict` function.

After running forward propagation, the network outputs a vector of probabilities. Your task:
1. Get the **final activation** (last element of `activations`) and convert it to a 1D array using `.A1`.
2. Find the **index of the maximum value** using `FindMaxActivation`.
3. Build a prediction vector of **zeros**, then set the element at that index to **1**.

> **Hint:** `FindMaxActivation` is already implemented below.

In [ ]:
def Predict(item, weights):
    layers = len(weights)
    item = np.append(1, item)  # Augment feature vector with bias

    activations = ForwardPropagation(item, weights, layers)

    ### YOUR CODE HERE ###
    # Step 1: Get final activation as 1D array: activations[-1].A1
    # Step 2: Find the index of the maximum activation using FindMaxActivation
    # Step 3: Create a zero vector of the same length as the output
    # Step 4: Set the element at the max index to 1
    # Step 5: Return the prediction vector
    outputFinal = None  # Replace this line
    ### END YOUR CODE ###


def FindMaxActivation(output):
    """Find the index of the maximum activation in the output vector"""
    m, index = output[0], 0
    for i in range(1, len(output)):
        if(output[i] > m):
            m, index = output[i], i
    return index

### ✏️ Exercise 6: Accuracy

Complete the `Accuracy` function.

For each sample in `X`:
1. Use `Predict` to get the network's prediction.
2. Compare it to the true label `y` (convert `Y[i]` to a list first).
3. If the prediction matches, increment the `correct` counter.
4. Return the **fraction** of correct predictions.

> **Hint:** `list(Y[i])` converts the one-hot row to a Python list for easy comparison.

In [ ]:
def Accuracy(X, Y, weights):
    """Run set through network, find overall accuracy"""
    correct = 0

    for i in range(len(X)):
        x, y = X[i], list(Y[i])
        ### YOUR CODE HERE ###
        # Step 1: Use Predict(x, weights) to get the network's guess
        # Step 2: If the guess matches y, increment correct
        pass  # Replace this line
        ### END YOUR CODE ###

    return correct / len(X)

## 🚀 Train the Network

Now that all functions are implemented, run the network! We use:
- **4 input features** (sepal/petal length & width)
- **2 hidden layers** with 5 and 10 nodes
- **3 output nodes** (one per class)
- **100 epochs**, **learning rate = 0.15**

Accuracy will be printed every 20 epochs.

In [ ]:
f = len(X[0])  # Number of features
o = len(Y[0])  # Number of outputs / classes

layers = [f, 5, 10, o]  # Number of nodes in each layer
lr, epochs = 0.15, 100

weights = NeuralNetwork(X_train, Y_train, X_val, Y_val, epochs=epochs, nodes=layers, lr=lr)

## 🧪 Test the Network

Finally, evaluate your trained network on the **test set**:

In [ ]:
print("Testing Accuracy: {}".format(Accuracy(X_test, Y_test, weights)))

## 🎉 Well Done!

You have built a neural network from scratch and trained it to classify Iris species.

Once you are done, compare your implementation to the solution notebook (`06_Building_NN_From_Scratch_Advanced_solution.ipynb`).

**Bonus challenges:**
- Try different layer sizes (e.g., `[f, 8, 8, o]`) and see how accuracy changes.
- Try different learning rates or number of epochs.
- Replace the Sigmoid activation with ReLU and observe the difference.